# Sweep Runner

This notebook expands the YAML sweep grids into concrete experiment configs, then runs them locally or submits them to SLURM. The CLI currently runs one experiment config at a time, so the notebook materializes one config per parameter combination.

`jepa.num_jepa_layers: 0` is included as a baseline-equivalent option. In a full factorial grid it will repeat across JEPA weights, dimensions, horizons, and modes; keep that if you want a strict full grid, or deduplicate those rows later.

In [ ]:
from copy import deepcopy
from itertools import product
from pathlib import Path
import subprocess

import yaml

from ablation_study_jepa.config.loader import load_config_dict
from ablation_study_jepa.config.schemas import ExperimentConfig

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sweep_path = root / "configs/sweeps/jepa_lejepa.yaml"  # or jepa_contrastive.yaml
max_runs = None  # set to a small integer for a dry run
generated_root = root / "configs/generated_sweeps"


In [ ]:
def set_dotted(config: dict, dotted_key: str, value):
    node = config
    parts = dotted_key.split(".")
    for part in parts[:-1]:
        node = node.setdefault(part, {})
    node[parts[-1]] = value


def iter_grid(sweep: dict):
    fixed = {}
    variable = {}
    for key, spec in sweep["parameters"].items():
        if key == "config":
            continue
        if "values" in spec:
            variable[key] = spec["values"]
        else:
            fixed[key] = spec["value"]

    keys = list(variable)
    for values in product(*(variable[key] for key in keys)):
        yield {**fixed, **dict(zip(keys, values))}


def materialize_sweep(sweep_path: Path, limit: int | None = None) -> list[Path]:
    sweep = yaml.safe_load(sweep_path.read_text())
    base_path = root / sweep["parameters"]["config"]["value"]
    base_config = load_config_dict(base_path)
    output_dir = generated_root / sweep_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    paths = []
    for index, overrides in enumerate(iter_grid(sweep)):
        if limit is not None and index >= limit:
            break
        config = deepcopy(base_config)
        for dotted_key, value in overrides.items():
            set_dotted(config, dotted_key, value)

        if config["jepa"]["num_jepa_layers"] == 0:
            config["jepa"]["enabled"] = False
            config["jepa"]["layer_selection_mode"] = "none"
        else:
            config["jepa"]["enabled"] = True
            if config["jepa"].get("layer_selection_mode") == "none":
                config["jepa"]["layer_selection_mode"] = "last_L"

        config["run_name"] = f"{base_config.get('run_name', sweep_path.stem)}_{index:04d}"
        config.setdefault("logging", {}).setdefault("wandb", {}).setdefault("tags", [])
        config["logging"]["wandb"]["tags"] = [
            *config["logging"]["wandb"]["tags"],
            sweep_path.stem,
            f"sweep_{index:04d}",
        ]

        ExperimentConfig.model_validate(config)
        path = output_dir / f"run_{index:04d}.yaml"
        path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
        paths.append(path)

    return paths


config_paths = materialize_sweep(sweep_path, limit=max_runs)
len(config_paths), config_paths[:3]


## Run Locally

Use this on a workstation or interactive GPU node. The experiment config controls `training.accelerator`, so leave it as `auto` or set it in the base config before materializing the sweep.

In [ ]:
run_local = False

if run_local:
    for config_path in config_paths:
        print(f"Running {config_path}", flush=True)
        subprocess.run(
            ["uv", "run", "ablation-study-jepa", "run", "--config", str(config_path)],
            cwd=root,
            check=True,
        )


## Submit To SLURM

This writes one `sbatch` command per generated config. Review the generated shell script, then run it from this notebook or from a login node.

In [ ]:
submit_script = generated_root / sweep_path.stem / "submit_slurm.sh"
lines = ["#!/bin/bash", "set -euo pipefail", ""]
for config_path in config_paths:
    lines.append(f"sbatch {root / 'slurm/run_experiment.sbatch'} {config_path}")
submit_script.write_text("\n".join(lines) + "\n", encoding="utf-8")
submit_script


In [ ]:
submit_to_slurm = False

if submit_to_slurm:
    subprocess.run(["bash", str(submit_script)], cwd=root, check=True)
